# **PENYORTIRAN DATASET, LABEL OTOMATIS, VALIDASI, DAN SPLIT 80/20**
### Komentar YouTube Laga Portugal vs Kroasia

Alur:

| Tahap | Isi |
|---|---|
| 1-2 | Import & muat `youtube_comments.csv` (upload otomatis) |
| 3 | Pembersihan teks — URL, mention, hashtag, timestamp, duplikat, < 3 kata |
| 4 | Filter Bahasa Indonesia |
| 5 | Filter relevansi topik (laga/VAR) |
| 6 | **Audit data dibuang** — semua baris yang dibuang diekspor + alasannya |
| 7 | **Label otomatis** — IndoBERT HuggingFace, fallback leksikon |
| 8 | **Sampel validasi 50/50/50** untuk dicek manual |
| 9 | **Hitung akurasi label** (per kelas + tertimbang) |
| 10 | Visualisasi — terpisah per cell |
| 11-12 | Split 80/20 & ekspor CSV |

Tanpa Excel, tanpa anotator. Manual hanya 150 baris untuk validasi.


# **1. IMPORT LIBRARY**


In [ ]:
!pip install -q openpyxl wordcloud pandas numpy matplotlib seaborn scikit-learn

import os, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Library siap.")


# **2. LOAD / UPLOAD DATASET MENTAH**


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

for p in ['youtube_comments.csv', '/content/youtube_comments.csv',
          '/home/alstear/Unduhan/youtube_comments.csv']:
    raw_data_path = p if os.path.exists(p) else None
    if raw_data_path:
        break

if raw_data_path is None:
    if IN_COLAB:
        from google.colab import files
        print("Upload 'youtube_comments.csv':")
        up = files.upload()
        raw_data_path = [f for f in up if f.endswith('.csv')][0]
    else:
        raise FileNotFoundError("File 'youtube_comments.csv' tidak ditemukan.")

print("Memuat:", raw_data_path)
df_raw = pd.read_csv(raw_data_path)
print("Baris mentah:", len(df_raw))
print("Kolom:", df_raw.columns.tolist())
display(df_raw.head())


# **3. PEMBERSIHAN TEKS**
Hapus URL, mention, hashtag, timestamp video (`8:07`), angka, tanda baca. Lalu buang komentar < 3 kata dan duplikat.


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'^\d{1,2}:\d{2}$', '', text.strip())
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

text_col = 'comment' if 'comment' in df_raw.columns else df_raw.columns[0]
df_raw['cleaned_text'] = df_raw[text_col].apply(clean_text)
df_raw['n_kata'] = df_raw['cleaned_text'].str.split().str.len()

df_clean = df_raw[df_raw['n_kata'] >= 3].copy()
df_clean = df_clean.drop_duplicates(subset=['cleaned_text']).copy()

print("Mentah            :", len(df_raw))
print("Lolos cleaning    :", len(df_clean))
print("Dibuang di tahap  :", len(df_raw) - len(df_clean))


# **4. PENYORTIRAN BAHASA INDONESIA (DUA SISI)**
Bukan lagi aturan "minimal 2 kata kunci" — aturan lama itu membuang 85% komentar Indonesia pendek.

Sekarang membandingkan dua sisi:
* **penanda Indonesia** (`yang`, `gak`, `banget`, `wasit`, `gol`, ...)
* **penanda asing** (Inggris/Portugis/Spanyol: `the`, `and`, `nao`, `muito`, `pero`, ...)

Lolos kalau ada penanda Indonesia **dan** porsinya >= 40% dari total penanda. Nama tim/pemain sengaja **tidak** dimasukkan ke daftar asing karena netral bahasa.


In [ ]:
ID_MARK = set("""
yang di dan ini itu dengan untuk tidak ada dari bisa sama gua gue lu lo gw aja banget bgt
udah udh kalo kl tapi tp bukan jangan jgn kok sih kan nya dia mereka kita kami lagi lg emang
bener ya nih dong gitu tau tahu mau kenapa knp karena krn jadi jd masih msh pasti pst pas saat
ketika harus hrs lebih lbh bang bg om mas bro cuy anjir njir cok wkwk wkwkwk haha hahaha
kasian kasihan beban jujur parah bagus mantap keren sedih kecewa lawan tanding menit babak
wasit penalti pinalti gol goal menang kalah main bola curang anulir offside pildun piala dunia
timnas portugal kroasia orang buat juga sudah belum akan aku saya kamu kalian semua sangat
sekali hanya saja kalau bila bahwa oleh pada kepada dalam antara tanpa sampai setelah sebelum
selama karena sehingga tetapi namun meski walau bilang kata cerita dukung benci suka salut
selamat terima kasih permainan pemain tim laga pertandingan kualitas hebat mantul gacor juara
final grup skor tendangan gawang peluit kartu hakim banyak sedikit terlalu cukup lebih kurang
hampir pernah tidak bukan belum jangan layak settingan diatur bantu untung rugi respect respek
lucu serem seru drama tegang kocak gila pinter bodoh tolol bego bloon bangsat asu jancok luar
dalam atas bawah sini sana begini begitu gimana bagaimana siapa kapan dimana tanding
""".split())

FOREIGN_MARK = set("""
the and you is are was were been being have has had do does did not no yes but or if then than
this that these those there their them they we he she it its his her our your my mine yours ours
theirs what which who whom whose when where why how all any some every each few many much more
most other another such only own same very too also just now here about above below over under
again further once because during before after between into through against from up down out off
on at by for with within without would could should may might must shall will can cannot
nao não voce você muito mais meu minha teu tua aqui isso isto esse essa este esta mas bem foi ser
ter tem vai pelo pela dos das uma uns umas com por que quem qual quando como onde porque tambem
também ja já pero cuando donde muy todo toda todos todas nada esto ese hay para gracias tambien
vez veces juego partido jugador equipo arbitro arbitraje
""".split())

def is_indonesian(text):
    tokens = re.findall(r'[a-zA-Z]+', text.lower())
    if not tokens:
        return False
    n_id = sum(1 for t in tokens if t in ID_MARK)
    n_asing = sum(1 for t in tokens if t in FOREIGN_MARK)
    if n_id == 0:
        return False
    return n_id / (n_id + n_asing) >= 0.4

df_indo = df_clean[df_clean['cleaned_text'].apply(is_indonesian)].copy()

print("Sebelum filter bahasa :", len(df_clean))
print("Setelah filter bahasa :", len(df_indo))
print("Dibuang               :", len(df_clean) - len(df_indo))


# **5. FILTER RELEVANSI TOPIK (TOLERAN TYPO)**
Aturan lama pakai daftar kata kunci persis, sehingga `Krosia`, `Pinalty`, `ofsaid`, `goll` tidak dikenali dan komentarnya ikut terbuang. Sekarang pakai **pola regex** yang menangkap varian typo dan nama pemain.

Contoh yang sekarang tertangkap: `kroa|krosi|croat`, `pinal|penal`, `of+side|ofsaid`, `go+l+`, `leao`, `ramos`, `dodo`, `costa`, `gvardiol`.

Komentar di luar topik (iklan TVRI, jadwal, spam) tetap dibuang.


In [ ]:
TOPIC_PATTERNS = [
    # tim & varian typo
    r'\bkroa\w*', r'\bkrosi\w*', r'\bcroat\w*', r'\bportu\w*',
    # perangkat & kejadian laga
    r'\bwasit\w*', r'\bvar\b', r'\bhakim\w*', r'\bgaris\b', r'\bpeluit\w*',
    r'\bkartu\w*', r'\bfifa\b', r'\buefa\b',
    r'\bpinal\w*', r'\bpenal\w*', r'\bof+side\w*', r'\bofsaid\w*', r'\boffsid\w*',
    r'\bgo+l+\w*', r'\banul\w*', r'\bdianul\w*', r'\bpelanggaran\w*',
    r'\btendangan\w*', r'\bgawang\w*', r'\bbola\w*', r'\bkepala\w*',
    r'\bsentuh\w*', r'\btangan\w*',
    # konteks turnamen
    r'\bskor\w*', r'\bbabak\w*', r'\bmenit\w*', r'\blaga\w*', r'\bpertandingan\w*',
    r'\bpildun\w*', r'\bpiala\w*', r'\bdunia\w*', r'\bgrup\w*', r'\bkualifikasi\w*',
    # nama pemain
    r'\bronaldo\w*', r'\bcr7\b', r'\bmodric\w*', r'\bbruno\w*', r'\bkovacic\w*',
    r'\bperisic\w*', r'\blivakovic\w*', r'\bpepe\b', r'\bdias\b', r'\bcancelo\w*',
    r'\bgvardiol\w*', r'\bleao\w*', r'\bramos\w*', r'\bdodo\w*', r'\bcosta\w*',
    r'\bfelix\w*', r'\bbrozovic\w*', r'\bkramaric\w*', r'\bmatic\w*',
    # hasil & suasana
    r'\bmenang\w*', r'\bkemenang\w*', r'\bkalah\w*', r'\bjuara\w*',
    r'\btim\b', r'\blawan\w*', r'\breplay\w*', r'\bpensiun\w*', r'\blegend\w*', r'\blegenda\w*',
    r'\bselebrasi\w*', r'\bstadion\w*', r'\bpendukung\w*', r'\bsuporter\w*',
    # penilaian khas
    r'\bcurang\w*', r'\badil\w*', r'\brugi\w*', r'\bdiuntungkan\w*', r'\bkecewa\w*',
    r'\bbangga\w*', r'\bdukung\w*', r'\bbela\w*', r'\blayak\w*', r'\bsettingan\w*',
    r'\bdiatur\w*', r'\bsetting\w*', r'\bsemifinal\w*', r'\bdibantu\b', r'\bbantu\w*', r'\bkontroversi\w*',
]

TOPIC_RE = re.compile('|'.join(TOPIC_PATTERNS))

def is_on_topic(text):
    return bool(TOPIC_RE.search(text.lower()))

df_topic = df_indo[df_indo['cleaned_text'].apply(is_on_topic)].copy()
df_topic['id'] = range(1, len(df_topic) + 1)

print("Sebelum filter topik :", len(df_indo))
print("Setelah filter topik :", len(df_topic))
print("Dibuang              :", len(df_indo) - len(df_topic))





# **6. AUDIT DATA DIBUANG (SEMUA, BUKAN SAMPEL)**
Semua baris yang dibuang diekspor lengkap dengan alasannya, supaya bisa diperiksa satu per satu di Excel.

Kolom `alasan`:
* `kurang dari 3 kata`
* `duplikat`
* `bukan bahasa indonesia`
* `di luar topik`

Berguna untuk memastikan filter tidak salah buang. Kalau ternyata ada komentar Indonesia bagus ikut terbuang, perbaiki filternya lalu ulangi.


In [ ]:
# Kumpulkan semua yang dibuang beserta alasannya
pendek = df_raw[df_raw['n_kata'] < 3].copy()
pendek['alasan'] = 'kurang dari 3 kata'

cukup = df_raw[df_raw['n_kata'] >= 3]
duplikat = cukup[cukup.duplicated(subset=['cleaned_text'], keep='first')].copy()
duplikat['alasan'] = 'duplikat'

bukan_indo = df_clean[~df_clean.index.isin(df_indo.index)].copy()
bukan_indo['alasan'] = 'bukan bahasa indonesia'

luar_topik = df_indo[~df_indo.index.isin(df_topic.index)].copy()
luar_topik['alasan'] = 'di luar topik'

kolom_audit = ['cleaned_text', 'alasan']
df_dibuang = pd.concat([
    pendek[kolom_audit],
    duplikat[kolom_audit],
    bukan_indo[kolom_audit],
    luar_topik[kolom_audit],
], ignore_index=True)

df_dibuang.to_csv('dataset_dibuang.csv', index=False)
df_topic[['id', 'cleaned_text']].to_csv('dataset_diloloskan.csv', index=False)

print("=" * 55)
print("REKAP PENYORTIRAN")
print("=" * 55)
print("Mentah                    :", len(df_raw))
print("Dibuang - kurang 3 kata   :", len(pendek))
print("Dibuang - duplikat        :", len(duplikat))
print("Dibuang - bukan Indonesia :", len(bukan_indo))
print("Dibuang - di luar topik   :", len(luar_topik))
print("TOTAL dibuang             :", len(df_dibuang))
print("LOLOS (siap dilabeli)     :", len(df_topic))
print("=" * 55)
print("")
print("File 'dataset_dibuang.csv' berisi SEMUA baris buangan + alasannya.")
print("Silakan periksa di Excel, lalu perbaiki filter kalau ada yang salah buang.")

display(df_dibuang.sample(min(10, len(df_dibuang)), random_state=SEED))


# **7. LABEL OTOMATIS**
Melabeli sentimen tanpa anotasi manual:

1. **IndoBERT dari HuggingFace** (`w11wo/indonesian-roberta-base-sentiment-classifier`) — 3 kelas, gratis, tanpa langganan. Butuh internet.
2. **Fallback leksikon** — kalau model gagal diunduh.

Sarkasme ditandai terpisah dengan penanda heuristik (`wkwk`, `katanya`, `sok`, dan sejenisnya). Ini **bukan** deteksi sarkasme sungguhan — di paper tulis sebagai "penanda heuristik", bukan "klasifikasi sarkasme".


In [ ]:
POS_WORDS = set(['bagus','mantap','keren','hebat','terbaik','senang','bangga','gacor','juara',
                 'respect','adil','puas','menang','berkelas','top','suka','layak','pantas',
                 'benar','tepat','jujur','bersih','indah','solid','stabil','luar biasa'])
NEG_WORDS = set(['curang','beban','buruk','kecewa','parah','kalah','rugi','anulir','rusak',
                 'bodoh','tolol','sial','kesal','lemah','buyar','hancur','malu','gagal',
                 'bias','berat','sewenang','kacau','jelek','payah','bancakan'])
SARKAS_MARKERS = set(['wkwk','wkwkwk','haha','hahaha','katanya','konon','sok','pantesan',
                      'cik','hadeh','heleh','cih','huh'])

def label_leksikon(teks):
    tok = teks.lower().split()
    p = sum(1 for w in tok if w in POS_WORDS)
    n = sum(1 for w in tok if w in NEG_WORDS)
    if p > n:
        return 'positif'
    if n > p:
        return 'negatif'
    return 'netral'

def label_sarkas(teks):
    tok = set(teks.lower().split())
    return 'ya' if (tok & SARKAS_MARKERS) else 'tidak'

teks_list = df_topic['cleaned_text'].tolist()
METODE_LABEL = 'Leksikon'
label_hasil = None

try:
    from transformers import pipeline
    import torch

    pipe = pipeline(
        'sentiment-analysis',
        model='w11wo/indonesian-roberta-base-sentiment-classifier',
        device=0 if torch.cuda.is_available() else -1,
        truncation=True,
        max_length=128,
    )
    keluaran = pipe(teks_list, batch_size=32)

    peta = {'positive': 'positif', 'neutral': 'netral', 'negative': 'negatif',
            'positif': 'positif', 'netral': 'netral', 'negatif': 'negatif'}
    label_hasil = [peta.get(k['label'].lower(), 'netral') for k in keluaran]
    METODE_LABEL = 'IndoBERT HuggingFace (w11wo/indonesian-roberta-base-sentiment-classifier)'
except Exception as e:
    print("Model HuggingFace gagal dipakai:", type(e).__name__, str(e)[:200])
    print("Beralih ke leksikon.")

if label_hasil is None:
    label_hasil = [label_leksikon(t) for t in teks_list]

df_labeled = df_topic.copy()
df_labeled['sentiment'] = label_hasil
df_labeled['sarkasme'] = df_labeled['cleaned_text'].apply(label_sarkas)

print("")
print("Metode pelabelan :", METODE_LABEL)
print("Jumlah data      :", len(df_labeled))
print("")
print("Distribusi sentimen:")
print(df_labeled['sentiment'].value_counts())
print("")
print("Sarkasme (heuristik):", int((df_labeled['sarkasme'] == 'ya').sum()), "komentar")


# **8. SAMPEL VALIDASI 50/50/50**
Mengambil **50 komentar per kelas** (total 150) untuk dicek manual.

Kenapa berimbang, bukan acak 120:
* Sampel acak 120 dari 2.000 hanya memberi ~13 komentar negatif — terlalu sedikit untuk menilai kelas minoritas.
* 50/50/50 memastikan tiap kelas dinilai dengan cukup.

**Catatan penting:** akurasi dari sampel berimbang **tidak boleh** dilaporkan langsung sebagai akurasi dataset. Harus ditimbang kembali dengan proporsi kelas sebenarnya (dihitung otomatis di cell 9).

File keluaran: `validasi_label_manual.csv` — isi kolom `label_manual` saja.


In [ ]:
N_PER_KELAS = 50

bagian = []
for kls in ['positif', 'netral', 'negatif']:
    subset = df_labeled[df_labeled['sentiment'] == kls]
    ambil = min(N_PER_KELAS, len(subset))
    if ambil > 0:
        bagian.append(subset.sample(n=ambil, random_state=SEED))
    print(f"{kls:8}: tersedia {len(subset):5}, diambil {ambil}")

df_validasi = pd.concat(bagian).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
df_validasi = df_validasi[['id', 'cleaned_text', 'sentiment', 'sarkasme']].copy()
df_validasi.columns = ['id', 'komentar', 'label_otomatis', 'sarkasme']
df_validasi['label_manual'] = ''
df_validasi['catatan'] = ''

df_validasi.to_csv('validasi_label_manual.csv', index=False)
df_validasi.to_excel('validasi_label_manual.xlsx', index=False)

print("")
print("Sampel validasi :", len(df_validasi), "komentar")
print("File           : validasi_label_manual.csv / .xlsx")
print("")
print("LANGKAH MANUAL:")
print("1. Buka file, isi kolom 'label_manual' dengan positif / netral / negatif")
print("2. JANGAN lihat kolom 'label_otomatis' saat menilai (tutup kolomnya dulu)")
print("3. Simpan, upload balik, jalankan cell 9")
print("")
display(df_validasi.head(5))


# **9. HITUNG AKURASI LABEL OTOMATIS**
Jalankan setelah `validasi_label_manual.csv` terisi.

Menghitung:
* **Akurasi per kelas** — seberapa bagus label otomatis di kelas positif, netral, negatif
* **Akurasi tertimbang** — angka inilah yang dilaporkan di paper (ditimbang dengan proporsi kelas sebenarnya)
* **Cohen's Kappa** — kesepakatan label otomatis vs manusia, dikoreksi kebetulan

Tafsiran Kappa: `<0,20` buruk · `0,21-0,40` sedang · `0,41-0,60` cukup · `0,61-0,80` baik · `>0,80` sangat baik.


In [ ]:
def cohen_kappa(a, b, labels):
    idx = {l: i for i, l in enumerate(labels)}
    m = np.zeros((len(labels), len(labels)), dtype=float)
    for x, y in zip(a, b):
        if x in idx and y in idx:
            m[idx[x], idx[y]] += 1
    total = m.sum()
    if total == 0:
        return 0.0
    po = np.trace(m) / total
    pe = (m.sum(axis=0) * m.sum(axis=1)).sum() / (total ** 2)
    return float((po - pe) / (1 - pe)) if pe < 1 else 0.0

def tafsir(k):
    if k < 0.20: return 'buruk'
    if k < 0.40: return 'sedang'
    if k < 0.60: return 'cukup'
    if k < 0.80: return 'baik'
    return 'sangat baik'

KELAS = ['positif', 'netral', 'negatif']
FILE_VALIDASI = 'validasi_label_manual.csv'
AKURASI_VALIDASI = None

if os.path.exists(FILE_VALIDASI):
    dfv = pd.read_csv(FILE_VALIDASI)
    dfv['label_manual'] = dfv['label_manual'].astype(str).str.lower().str.strip()
    dfv.loc[~dfv['label_manual'].isin(KELAS), 'label_manual'] = np.nan

    terisi = dfv.dropna(subset=['label_manual'])
    print("Baris validasi  :", len(dfv))
    print("Sudah diisi     :", len(terisi))

    if len(terisi) >= 20:
        print("")
        print("AKURASI PER KELAS (label otomatis vs manual)")
        print("-" * 52)
        akurasi_kelas = {}
        for k in KELAS:
            sub = terisi[terisi['label_otomatis'] == k]
            if len(sub) == 0:
                continue
            benar = (sub['label_manual'] == k).sum()
            acc = benar / len(sub)
            akurasi_kelas[k] = acc
            print(f"{k:8}: {benar:3}/{len(sub):3} benar = {acc*100:5.1f}%")

        proporsi = df_labeled['sentiment'].value_counts(normalize=True).to_dict()
        akurasi_tertimbang = sum(proporsi.get(k, 0) * akurasi_kelas.get(k, 0)
                                 for k in KELAS)
        akurasi_mentah = (terisi['label_otomatis'] == terisi['label_manual']).mean()
        kappa = cohen_kappa(terisi['label_otomatis'], terisi['label_manual'], KELAS)

        print("")
        print("-" * 52)
        print(f"Akurasi sampel (50/50/50)   : {akurasi_mentah*100:.1f}%  <- jangan dilaporkan")
        print(f"Akurasi tertimbang (dilapor): {akurasi_tertimbang*100:.1f}%")
        print(f"Cohen's Kappa               : {kappa:.4f} ({tafsir(kappa)})")

        AKURASI_VALIDASI = akurasi_tertimbang

        silang = pd.crosstab(terisi['label_manual'], terisi['label_otomatis'])
        print("")
        print("Tabel silang (baris=manual, kolom=otomatis):")
        print(silang)

        plt.figure(figsize=(5, 4))
        sns.heatmap(silang, annot=True, fmt='d', cmap='Blues')
        plt.title('Label Manual vs Label Otomatis', fontweight='bold')
        plt.ylabel('Manual')
        plt.xlabel('Otomatis')
        plt.tight_layout()
        plt.show()
    else:
        print("")
        print("Belum cukup data terisi (minimal 20). Isi 'validasi_label_manual.csv' dulu.")
else:
    print("File", FILE_VALIDASI, "belum ada.")
    print("Jalankan cell 8 dulu, isi kolom 'label_manual', lalu ulangi cell ini.")


# **10. VISUALISASI**
Terpisah per cell. Kalau validasi label belum selesai, grafik di bawah tetap memakai label otomatis — tulis di paper bahwa label bersifat otomatis sampai divalidasi.


### **10.1 Diagram Batang Distribusi Sentimen**


In [ ]:
plt.figure(figsize=(8, 5))
counts = df_labeled['sentiment'].value_counts()
warna = {'negatif': '#e74c3c', 'netral': '#95a5a6', 'positif': '#2ecc71'}
bar_colors = [warna.get(s, '#3498db') for s in counts.index]

bars = plt.bar(counts.index, counts.values, color=bar_colors, edgecolor='black', alpha=0.85)
plt.title('Distribusi Kelas Sentimen Komentar YouTube', fontsize=13, fontweight='bold')
plt.xlabel('Sentimen')
plt.ylabel('Jumlah Komentar')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars:
    y = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, y + 5,
             str(y) + ' (' + format(y / len(df_labeled) * 100, '.1f') + '%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()


### **10.2 Diagram Donat Proporsi Sentimen**


In [ ]:
plt.figure(figsize=(7, 7))
counts = df_labeled['sentiment'].value_counts()
warna = {'negatif': '#e74c3c', 'netral': '#95a5a6', 'positif': '#2ecc71'}
pie_colors = [warna.get(s, '#3498db') for s in counts.index]

plt.pie(counts.values, labels=counts.index, autopct='%1.1f%%', pctdistance=0.75,
        colors=pie_colors, startangle=140,
        textprops={'fontsize': 11, 'fontweight': 'bold'})
plt.gca().add_artist(plt.Circle((0, 0), 0.55, fc='white'))
plt.title('Proporsi Sentimen Komentar YouTube', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### **10.3 Proporsi Sarkasme per Kelas Sentimen**


In [ ]:
tabel_sarkas = pd.crosstab(df_labeled['sentiment'], df_labeled['sarkasme'], normalize='index') * 100
tabel_sarkas.plot(kind='bar', stacked=True, figsize=(8, 5),
                  color=['#3498db', '#f39c12'], edgecolor='black', alpha=0.85)
plt.title('Proporsi Penanda Sarkasme per Kelas Sentimen', fontsize=13, fontweight='bold')
plt.xlabel('Sentimen')
plt.ylabel('Persentase (%)')
plt.legend(title='Sarkasme', labels=['Tidak', 'Ya'])
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('Penanda sarkasme:', int((df_labeled['sarkasme'] == 'ya').sum()), 'komentar',
      '(' + str(round((df_labeled['sarkasme'] == 'ya').mean() * 100, 2)) + '%)')


### **10.4 Distribusi Panjang Kata per Sentimen**


In [ ]:
df_labeled['word_count'] = df_labeled['cleaned_text'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_labeled, x='sentiment', y='word_count',
            palette={'negatif': '#e74c3c', 'netral': '#95a5a6', 'positif': '#2ecc71'})
plt.title('Distribusi Jumlah Kata per Kelas Sentimen', fontsize=13, fontweight='bold')
plt.xlabel('Sentimen')
plt.ylabel('Jumlah Kata')
plt.ylim(0, df_labeled['word_count'].quantile(0.98))
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(df_labeled.groupby('sentiment')['word_count'].describe().round(2))


### **10.5 Word Cloud - Sentimen POSITIF**


In [ ]:
custom_stopwords = set([
    'yang','di','dan','ini','itu','dengan','untuk','tidak','ada','dari','bisa','sama',
    'gua','gue','lu','lo','gw','aja','banget','bgt','udah','udh','kalo','kl','tapi','tp',
    'bukan','jangan','jgn','kok','sih','kan','nya','dia','mereka','kita','kami','lagi',
    'lg','emang','bener','ya','nih','dong','gitu','tau','tahu','mau','kenapa','knp',
    'karena','krn','jadi','jd','masih','msh','pasti','pst','pas','saat','ketika','harus',
    'hrs','lebih','lbh','bang','bg','om','mas','bro','cuy','wkwk','wkwkwk','haha',
    'hahaha','deh','lah','tuh','kek','mah'
])

pos_corpus = ' '.join(df_labeled[df_labeled['sentiment'] == 'positif']['cleaned_text'])
if pos_corpus.strip():
    wc = WordCloud(width=800, height=450, background_color='white', colormap='Greens',
                   stopwords=custom_stopwords, max_words=100, random_state=SEED).generate(pos_corpus)
    plt.figure(figsize=(10, 5.5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title('Word Cloud - Sentimen POSITIF', fontsize=14, fontweight='bold', color='#27ae60')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Tidak ada komentar positif.')


### **10.6 Word Cloud - Sentimen NEGATIF**


In [ ]:
neg_corpus = ' '.join(df_labeled[df_labeled['sentiment'] == 'negatif']['cleaned_text'])
if neg_corpus.strip():
    wc = WordCloud(width=800, height=450, background_color='white', colormap='Reds',
                   stopwords=custom_stopwords, max_words=100, random_state=SEED).generate(neg_corpus)
    plt.figure(figsize=(10, 5.5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title('Word Cloud - Sentimen NEGATIF', fontsize=14, fontweight='bold', color='#c0392b')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Tidak ada komentar negatif.')


### **10.7 Word Cloud - Sentimen NETRAL**


In [ ]:
neu_corpus = ' '.join(df_labeled[df_labeled['sentiment'] == 'netral']['cleaned_text'])
if neu_corpus.strip():
    wc = WordCloud(width=800, height=450, background_color='white', colormap='Blues',
                   stopwords=custom_stopwords, max_words=100, random_state=SEED).generate(neu_corpus)
    plt.figure(figsize=(10, 5.5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title('Word Cloud - Sentimen NETRAL', fontsize=14, fontweight='bold', color='#2980b9')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Tidak ada komentar netral.')


### **10.8 Word Cloud - Komentar Berpenanda Sarkasme**


In [ ]:
sarkas_corpus = ' '.join(df_labeled[df_labeled['sarkasme'] == 'ya']['cleaned_text'])
if sarkas_corpus.strip():
    wc = WordCloud(width=800, height=450, background_color='black', colormap='autumn',
                   stopwords=custom_stopwords, max_words=100, random_state=SEED).generate(sarkas_corpus)
    plt.figure(figsize=(10, 5.5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title('Word Cloud - Komentar Berpenanda Sarkasme', fontsize=14, fontweight='bold', color='#d35400')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Tidak ada komentar berpenanda sarkasme.')


# **11. SPLIT DATASET 80% TRAINING & 20% TESTING**


In [ ]:
X = df_labeled['cleaned_text']
y = df_labeled['sentiment']

X_train, X_test, idx_train, idx_test = train_test_split(
    X, df_labeled.index, test_size=0.20, random_state=SEED, stratify=y
)

train_df = df_labeled.loc[idx_train, ['cleaned_text', 'sentiment', 'sarkasme']].copy().reset_index(drop=True)
test_df = df_labeled.loc[idx_test, ['cleaned_text', 'sentiment', 'sarkasme']].copy().reset_index(drop=True)
train_df.columns = ['comment', 'sentiment', 'sarkasme']
test_df.columns = ['comment', 'sentiment', 'sarkasme']

print('=' * 55)
print('Total dataset  :', len(df_labeled), 'baris')
print('Training (80%) :', len(train_df), 'baris')
print('Testing  (20%) :', len(test_df), 'baris')
print('=' * 55)
print('')
print('Distribusi training (%):')
print((train_df['sentiment'].value_counts(normalize=True) * 100).round(2))
print('')
print('Distribusi testing (%):')
print((test_df['sentiment'].value_counts(normalize=True) * 100).round(2))


# **12. SIMPAN HASIL KE CSV**


In [ ]:
df_labeled[['cleaned_text', 'sentiment', 'sarkasme']].rename(
    columns={'cleaned_text': 'comment'}).to_csv('dataset_youtube_labeled_all.csv', index=False)
train_df.to_csv('train_dataset_80.csv', index=False)
test_df.to_csv('test_dataset_20.csv', index=False)

print('=' * 62)
print('SELESAI')
print('=' * 62)
print('Metode label :', METODE_LABEL)
if AKURASI_VALIDASI is not None:
    print('Akurasi tertimbang (validasi manual):', str(round(AKURASI_VALIDASI * 100, 1)) + '%')
else:
    print('Akurasi validasi: belum dihitung (isi validasi_label_manual.csv)')
print('')
print('1. dataset_youtube_labeled_all.csv :', len(df_labeled), 'baris')
print('2. train_dataset_80.csv            :', len(train_df), 'baris')
print('3. test_dataset_20.csv             :', len(test_df), 'baris')
print('4. dataset_dibuang.csv             :', len(df_dibuang), 'baris (audit buangan)')
print('5. validasi_label_manual.csv       :', len(df_validasi), 'baris (sampel cek)')
